In [ ]:
# ============================================
# Improved TinyML Multi-Word KWS
# 10 commands + unknown + silence
# Depthwise-Separable CNN (higher accuracy)
# ============================================

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# ===============================
# 1 Audio Parameters
# ===============================
FRAME_LENGTH = 512
FRAME_STEP = 256
FFT_LENGTH = FRAME_LENGTH
SAMPLE_RATE = 16000
TARGET_SAMPLES = SAMPLE_RATE

LOWER_HZ = 20.0
UPPER_HZ = 4000.0

N_MEL = 40
FIXED_FRAMES = 61

TARGET_WORDS = ["yes"]

In [ ]:
# ===============================
# 2 Load Dataset
# ===============================
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'speech_commands',
    split=['train', 'validation', 'test'],
    shuffle_files=True,
    with_info=True,
    as_supervised=True
)

class_names = ds_info.features['label'].names
target_ids = [class_names.index(w) for w in TARGET_WORDS]

NUM_CLASSES = len(TARGET_WORDS) + 2  # unknown + silence

In [ ]:
# ===============================
# 3 Audio Preprocessing
# ===============================
def fix_audio_length(audio):
    audio = audio[:TARGET_SAMPLES]
    padding = TARGET_SAMPLES - tf.shape(audio)[0]
    audio = tf.pad(audio, [[0, padding]])
    audio.set_shape([TARGET_SAMPLES])
    return audio

def compute_log_mel(audio, label=None):
    audio = tf.cast(audio, tf.float32) / 32768.0

    stft = tf.signal.stft(
        audio,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP,
        fft_length=FFT_LENGTH,
        window_fn=tf.signal.hann_window
    )

    power_spectrum = tf.abs(stft) ** 2
    num_spectrogram_bins = FFT_LENGTH // 2 + 1

    mel_weights = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MEL,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=LOWER_HZ,
        upper_edge_hertz=UPPER_HZ
    )

    mel_spectrogram = tf.tensordot(power_spectrum, mel_weights, axes=1)
    mel_spectrogram.set_shape(stft.shape[:-1].concatenate(mel_weights.shape[-1:]))
    log_mel = tf.math.log(mel_spectrogram + 1e-6)

    return log_mel, label

def pad_log_mel(log_mel):
    log_mel = log_mel[:FIXED_FRAMES, :]
    pad_len = FIXED_FRAMES - tf.shape(log_mel)[0]
    log_mel = tf.pad(log_mel, [[0, pad_len], [0, 0]])
    log_mel = tf.ensure_shape(log_mel, [FIXED_FRAMES, N_MEL])
    return log_mel

In [ ]:
# ===============================
# 4 Preprocessing + Augmentation
# ===============================

MAX_SHIFT = 1600  # 100ms at 16kHz

def time_shift(audio):
    shift = tf.random.uniform([], -MAX_SHIFT, MAX_SHIFT, dtype=tf.int32)
    return tf.roll(audio, shift, axis=0)

def random_volume(audio):
    scale = tf.random.uniform([], 0.8, 1.2)
    return audio * scale

def preprocess(audio, label, training=False):
    # 1. Fix audio length
    audio = fix_audio_length(audio)

    # Cast to float32 first
    audio = tf.cast(audio, tf.float32)

    # 2. Data augmentation (training only)
    if training:
        audio = time_shift(audio)
        audio = random_volume(audio)

    # 3. Compute log-Mel spectrogram
    log_mel, _ = compute_log_mel(audio)

    # 4. Pad or trim frames
    log_mel = pad_log_mel(log_mel)

    # 5. Normalize (mean/std) per example
    #mean = tf.reduce_mean(log_mel)
    #std  = tf.math.reduce_std(log_mel)
    #log_mel = (log_mel - mean) / (std + 1e-6)

    # 6. Add channel dimension
    log_mel = tf.expand_dims(log_mel, -1)  # shape: [FIXED_FRAMES, N_MEL, 1]

    # 7. Map label to target/unknown/silence
    label = tf.numpy_function(map_label, [label], tf.int64)
    label.set_shape([])

    return log_mel, label


In [ ]:
# ===============================
# 5 Label Mapping
# ===============================
silence_id = class_names.index("_silence_")

def map_label(label):
    # 0 = yes (target), 1 = unknown (other words), 2 = silence
    if label == silence_id:
        return len(TARGET_WORDS) + 1  # silence

    if label in target_ids:
        return target_ids.index(label)

    return len(TARGET_WORDS)  # unknown


BATCH_SIZE = 32

def preprocess_train(a, l):
    return preprocess(a, l, True)

def preprocess_eval(a, l):
    return preprocess(a, l, False)

ds_train = ds_train.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE) \
                   .shuffle(1000) \
                   .batch(BATCH_SIZE) \
                   .prefetch(tf.data.AUTOTUNE)

ds_val = ds_val.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE) \
               .batch(BATCH_SIZE) \
               .prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE) \
                 .batch(BATCH_SIZE) \
                 .prefetch(tf.data.AUTOTUNE)

In [ ]:

# ===============================
# 6 Improved TinyML Model (Recommended)
# ===============================
def build_kws_model(input_shape=(FIXED_FRAMES, N_MEL, 1),
                    num_classes=NUM_CLASSES):

    inputs = tf.keras.Input(shape=input_shape)

    # ---------------------------
    # Initial Conv
    # ---------------------------
    x = tf.keras.layers.Conv2D(
        32,
        (3,3),
        strides=(2,2),
        padding='same',
        use_bias=False
    )(inputs)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    # ---------------------------
    # Depthwise Block 1
    # ---------------------------
    x = tf.keras.layers.DepthwiseConv2D(
        (3,3),
        padding='same',
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(
        48,
        (1,1),
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    # ---------------------------
    # Depthwise Block 2
    # ---------------------------
    x = tf.keras.layers.DepthwiseConv2D(
        (3,3),
        padding='same',
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(
        64,
        (1,1),
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    # ---------------------------
    # Dropout Regularization
    # ---------------------------
    x = tf.keras.layers.Dropout(0.3)(x)

    # ---------------------------
    # Classifier (1x1 Conv)
    # ---------------------------
    x = tf.keras.layers.Conv2D(
        num_classes,
        (1,1),
        padding='same'
    )(x)

    # ---------------------------
    # Global Average Pool
    # ---------------------------
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    outputs = tf.keras.layers.Softmax()(x)

    return tf.keras.Model(inputs, outputs)

model = build_kws_model()
model.summary()

In [ ]:
# ===============================
# 7 Compile
# ===============================
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# ===============================
# 8 Train
# ===============================
model.fit(ds_train,
          validation_data=ds_val,
          epochs=3)

In [ ]:
# ===============================
# 9 Evaluate
# ===============================
model.evaluate(ds_test)

In [ ]:
# ===============================
# 10 Convert to Full INT8
# ===============================
def representative_dataset():
    for x, _ in ds_train.unbatch().take(1000):
        yield [tf.expand_dims(tf.cast(x, tf.float32), 0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("kws_1class_depthwise_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved kws_1class_depthwise_int8.tflite")

In [ ]:
mel_matrix = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins=N_MEL,
    num_spectrogram_bins=FFT_LENGTH // 2 + 1,
    sample_rate=SAMPLE_RATE,
    lower_edge_hertz=LOWER_HZ,
    upper_edge_hertz=UPPER_HZ
)

#mel_matrix = mel_matrix.numpy()  # convert to NumPy array

# Just remove any transpose
mel_matrix = mel_matrix.numpy().T  # shape = [N_MEL, FFT_LENGTH//2 + 1]

def print_c_array(matrix, name="melFilterbank"):
    rows, cols = matrix.shape  # rows = N_MEL, cols = FFT bins
    print(f"float {name}[{rows}][{cols}] = {{")
    for r in range(rows):
        print("  {", end="")
        print(", ".join(f"{matrix[r, c]:.6f}" for c in range(cols)), end="")
        print("},")
    print("};")

print_c_array(mel_matrix, name="melFilterbank")